# SENTINEL — Training an AI-Safety Overseer on Colab T4

**Hackathon submission • runs end-to-end on a free Tesla T4 GPU.**

This notebook trains a 1.7B-parameter "Overseer" model that reviews actions
proposed by an autonomous incident-response agent and decides whether each
action is safe to execute. Training uses **GRPO** (Group Relative Policy
Optimization, [DeepSeekMath, 2024](https://arxiv.org/abs/2402.03300))
against the [SENTINEL OpenEnv](https://huggingface.co/spaces/Elliot89/sentinel)
on Hugging Face Spaces.

## What you get when you `Runtime → Run all`

| Phase | What happens | Wall clock on T4 |
|---|---|---|
| 1.1 | Clone the SENTINEL repo | ~10 s |
| 1.2 | Install pinned dependencies | ~6 min |
| 1.3 | Restart the runtime once (mandatory) | ~5 s |
| 1.4 | Re-bootstrap + verify imports + check GPU | ~5 s |
| 1.5 | Set training knobs | <1 s |
| 2   | Wake the SENTINEL OpenEnv on HF Spaces | ~60 s |
| 3   | Load Qwen3-1.7B in 4-bit + attach LoRA adapter | ~3 min |
| 3.1 | *(optional)* Zero-shot baseline F1 | ~10 min |
| 4   | SFT warmup — teach JSON output format | ~5 min |
| 5   | GRPO smoke test (5 steps) — gate the long run | ~6 min |
| 6   | GRPO training (60 steps default) | ~45 min |
| 7   | Eval trained adapter + render plots inline | ~10 min |
| 8   | *(optional)* Push LoRA adapter to your HF Hub repo | ~1 min |

> **Free Colab T4 = 16 GB VRAM, ~12 h cap.** This notebook stays well inside
> both. Every training knob is tunable in **Section 1.5** if you have a
> stronger runtime (L4 / A100 / H100).

## What this trains

The Overseer model sees one Responder-proposed action per turn (e.g. "scale
the postgres-primary deployment to 0 replicas") plus a counterfactual preview
of what would happen if executed, and outputs a decision JSON:

```json
{"decision": "block", "justification": "scale-to-zero on a primary DB without backup is catastrophic"}
```

Decisions are scored against a per-action ground-truth class
(`CATASTROPHIC` / `WRONG` / `CORRECT` / `AMBIGUOUS`) with severity-weighted
reward. The grader is in [`graders.py`](https://github.com/MrEinsteinE/sentinel-openenv/blob/main/graders.py).

## How to read this notebook

The training pipeline (model loading, SFT, GRPO loop, eval harness, plots,
auto-abort on stalled reward) lives in
[`training/grpo_hf_job.py`](https://github.com/MrEinsteinE/sentinel-openenv/blob/main/training/grpo_hf_job.py).
This notebook re-uses every function in that module so the Colab path and
the production HF Jobs path share **exactly one implementation** — no drift.

The cells below mostly call into that module; the markdown above each cell
explains what's happening inside the call.


## 1. Bootstrap

> ⚠️ **Run cells 1.1 → 1.4 in order.** Section 1.3 restarts the kernel —
> that's intentional and required (Colab pre-installs numpy 2.x and bumps
> some C extensions during install, so we need a clean Python process
> before importing Unsloth).

### 1.1 Clone the SENTINEL repo


In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys

REPO_URL = os.environ.get("GIT_REPO", "https://github.com/MrEinsteinE/sentinel-openenv")
REPO_DIR = pathlib.Path(os.environ.get("SENTINEL_WORKDIR", "/content/sentinel-openenv"))
BRANCH   = os.environ.get("GIT_BRANCH", "main")

if (REPO_DIR / ".git").exists():
    print(f"Repo already at {REPO_DIR} — skipping clone.")
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth=1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
    print(f"Cloned → {REPO_DIR}")

# Persist for the post-restart cell.
os.environ["SENTINEL_WORKDIR"]      = str(REPO_DIR)
os.environ["SENTINEL_SKIP_BOOTSTRAP"] = "1"
os.chdir(REPO_DIR)
for p in (str(REPO_DIR), str(REPO_DIR / "training")):
    if p not in sys.path:
        sys.path.insert(0, p)


### 1.2 Install pinned dependencies (one shot)

This block is **the verified-working install for Colab T4 in April 2026**.
The pin choices and ordering matter:

* **`numpy==2.0.2` with `--force-reinstall --no-cache-dir`** —
  Colab pre-installs numpy 2.x but during the rest of the install some
  packages corrupt the install (replacing `numpy/__init__.py` but leaving
  the old `numpy/random/mtrand.so`). Forcing a fresh download fixes the
  cryptic `numpy.dtype size changed (Expected 96, got 88)` error.
* **`vllm==0.9.2` + `triton==3.2.0`** — T4 (sm_75) cannot run newer
  vLLM, which requires sm_80+ and crashes at import.
* **`transformers==4.56.2`** — Unsloth 2026.4.4's allowed window.
* **`trl==0.21.0` with `--no-deps`** — we use TRL 0.21's stable callback
  API. `--no-deps` so it can't drag a newer transformers that breaks Unsloth.
* **`unsloth==2026.4.4` + `unsloth_zoo==2026.4.4`** — pinned together.
* peft, accelerate, datasets, huggingface_hub, bitsandbytes, xformers
  come along transitively from `unsloth`.

`%%capture` swallows the verbose pip output. Watch the spinner in the
cell's `[ ]` bracket — it takes ~6 minutes on a fresh T4.


In [ ]:
%%capture
# ── 1.2a Clean any partially-installed state from a previous run ─────────
# Crucial for kernels that already had a broken install attempt — pip will
# otherwise refuse to reinstall numpy because the version string looks fine.
!pip uninstall -y -qqq numpy peft datasets unsloth unsloth_zoo \
    bitsandbytes accelerate xformers
!pip cache purge

# ── 1.2b Install numpy ALONE first, with --force-reinstall --no-cache-dir.
# This is the line that prevents the dtype-size traceback. We pin to 2.0.2
# because that's what Colab's stack expects.
!pip install --no-cache-dir --force-reinstall -qqq "numpy==2.0.2"

# ── 1.2c Install everything else against the fresh numpy. We deliberately
# do NOT pin torch — xformers will pull whatever torch version it needs
# (currently 2.7.x). One install pass so pip's resolver sees the full
# constraint set at once.
!pip install --no-cache-dir -qqq \
    "vllm==0.9.2" "triton==3.2.0" \
    "xformers" "bitsandbytes" \
    "transformers==4.56.2" "accelerate" "peft" "datasets" "huggingface_hub" \
    "unsloth==2026.4.4" "unsloth_zoo==2026.4.4" \
    "torchvision"

# trl with --no-deps so it can't override our transformers/peft pins.
!pip install --no-cache-dir --no-deps -qqq "trl==0.21.0"

# SENTINEL's own light deps (matplotlib for plots, requests for the HTTP
# env client, pydantic for the data contracts in models.py).
!pip install --no-cache-dir -qqq matplotlib requests pydantic

# Tell Unsloth to use its memory-efficient vLLM standby mode (30% extra
# context length). Read by unsloth_zoo at import time.
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"


### 1.3 Restart the runtime once (mandatory)

The install above replaces several C extensions (numpy, bitsandbytes,
xformers). The currently-running Python process still has the *old* C
extensions loaded into memory. To pick up the new ones we need a fresh
Python interpreter.

The cell below restarts the Colab kernel automatically. **You'll see a
`SystemExit` traceback — that's expected; ignore it.** Colab will reconnect
to the same VM (your installed packages persist) within ~5 seconds.

The restart is **gated by an on-disk marker** so it only fires once per
session, even if you `Run all` again.


In [ ]:
import pathlib

_marker = pathlib.Path("/content/.sentinel_kernel_restarted")
if _marker.exists():
    print("Kernel was already restarted in this session — nothing to do.")
else:
    _marker.write_text("1")
    print("Restarting Colab kernel to pick up freshly-installed C extensions...")
    print("(Expected: SystemExit traceback. Wait ~5s for reconnect, then run Section 1.4.)")
    import IPython
    IPython.get_ipython().kernel.do_shutdown(restart=True)


### 1.4 Re-bootstrap and verify

After the kernel restart, this cell:

1. Re-establishes the working directory + `sys.path` (the kernel forgot them).
2. **Imports `unsloth` FIRST**, before any other ML library lands in
   `sys.modules`. Unsloth monkey-patches `transformers` at import time;
   if `transformers` is loaded first, Unsloth refuses to apply its
   optimizations and prints a warning.
3. Imports every package the rest of the notebook needs and prints its
   version, so any missing module fails loudly here instead of cryptically
   four cells later.
4. Verifies the GPU is a T4 (or stronger) and prints VRAM / compute capability.

You should see all `OK` lines plus `🦥 Unsloth …: Fast Qwen3 patching` in
the output — that's the smoking-gun signal that Unsloth installed cleanly.


In [ ]:
import os
import pathlib
import sys

# Re-establish working dir + sys.path (lost across kernel restart).
REPO_DIR = pathlib.Path(os.environ.get("SENTINEL_WORKDIR", "/content/sentinel-openenv"))
assert (REPO_DIR / ".git").exists(), \
    f"Repo missing at {REPO_DIR} after restart — re-run Section 1.1."
os.environ["SENTINEL_WORKDIR"]      = str(REPO_DIR)
os.environ["SENTINEL_SKIP_BOOTSTRAP"] = "1"
os.chdir(REPO_DIR)
for p in (str(REPO_DIR), str(REPO_DIR / "training")):
    if p not in sys.path:
        sys.path.insert(0, p)

# ── CRITICAL: import unsloth BEFORE anything that touches transformers. ─────
# Unsloth's monkey-patches don't apply if transformers is already loaded.
import unsloth                                    # noqa: F401

# Now everything else is safe to import.
import numpy, torch, transformers, trl, peft, accelerate
import datasets, huggingface_hub, bitsandbytes, xformers, vllm, triton
import unsloth_zoo

print("─── installed package versions ───────────────────────────────────────")
for pkg in [unsloth, unsloth_zoo, torch, numpy, transformers, trl, peft,
            accelerate, datasets, huggingface_hub, bitsandbytes, xformers,
            vllm, triton]:
    print(f"  ✓ {pkg.__name__:18s} {getattr(pkg, '__version__', '?')}")

# ── GPU check ────────────────────────────────────────────────────────────────
assert torch.cuda.is_available(), "No CUDA GPU. Runtime → Change runtime type → T4 GPU."
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
major, minor = torch.cuda.get_device_capability(0)
dtype_mode = "bf16-capable (Ampere+)" if major >= 8 else "fp16 only (Turing/T4)"

print()
print("─── GPU ─────────────────────────────────────────────────────────────")
print(f"  device          : {gpu_name}")
print(f"  VRAM            : {vram_gb:.1f} GB")
print(f"  compute capab.  : {major}.{minor}  ({dtype_mode})")
print(f"  CUDA            : {torch.version.cuda}")

print()
print("Bootstrap complete — proceed to Section 1.5.")


## 1.5 Configuration — every training knob

These environment variables are read by `training/grpo_hf_job.py` at
function-call time. Keeping them here means the Colab path and the HF Jobs
path share the same code — no forks.

### Defaults below are tuned for Colab T4 (16 GB VRAM, ~12 h cap)

| Knob | Default here | What it does |
|---|---|---|
| `SENTINEL_USE_VLLM` | `0` | Disable colocated vLLM. Doesn't fit in 16 GB alongside training; we use plain `model.generate(...)` instead. |
| `SENTINEL_GRPO_NUM_GENERATIONS` | `2` | GRPO group size. Smaller = less VRAM. (HF Jobs default: 4.) |
| `SENTINEL_GRPO_MAX_COMPLETION_LENGTH` | `256` | Cap on Overseer-decision tokens. Most decisions are <100 tokens. |
| `SENTINEL_GRPO_GRADIENT_ACCUMULATION_STEPS` | `4` | Effective batch = 4 × num_generations. |
| `SENTINEL_GRPO_MAX_STEPS` | `60` | T4 demo budget. Bump to 200+ on L4/A100. |
| `SENTINEL_GRPO_SAVE_STEPS` | `10` | Plot/checkpoint cadence. |
| `SENTINEL_GRPO_LOGGING_STEPS` | `1` | Reward logging cadence. |
| `SENTINEL_RUN_ZEROSHOT_EVAL` | unset | Skip the optional zero-shot baseline (set to `1` in Section 3.1 to enable). |

If you're on L4 / A100, comment out the T4 line and uncomment the L4/A100
preset block — the rest of the notebook will pick up the change.


In [ ]:
import os

# ── T4 (free Colab) defaults ──────────────────────────────────────────────────
os.environ["SENTINEL_USE_VLLM"]                         = "0"
os.environ["SENTINEL_GRPO_NUM_GENERATIONS"]             = "2"
os.environ["SENTINEL_GRPO_MAX_COMPLETION_LENGTH"]       = "256"
os.environ["SENTINEL_GRPO_GRADIENT_ACCUMULATION_STEPS"] = "4"
os.environ["SENTINEL_GRPO_MAX_STEPS"]                   = "60"
os.environ["SENTINEL_GRPO_SAVE_STEPS"]                  = "10"
os.environ["SENTINEL_GRPO_LOGGING_STEPS"]               = "1"

# ── L4 / A100 preset — uncomment if you upgraded the runtime ─────────────────
# os.environ["SENTINEL_USE_VLLM"]                         = "1"
# os.environ["SENTINEL_GRPO_NUM_GENERATIONS"]             = "4"
# os.environ["SENTINEL_GRPO_MAX_COMPLETION_LENGTH"]       = "512"
# os.environ["SENTINEL_GRPO_GRADIENT_ACCUMULATION_STEPS"] = "8"
# os.environ["SENTINEL_GRPO_MAX_STEPS"]                   = "400"

# ── Stable across runtimes ───────────────────────────────────────────────────
os.environ.setdefault("SENTINEL_URL", "https://elliot89-sentinel.hf.space")
os.environ.setdefault("MODEL_NAME",   "unsloth/Qwen3-1.7B")
os.environ.setdefault("MODEL_REPO",   "Elliot89/sentinel-overseer-qwen3-1.7b")

# Pull HF_TOKEN from Colab Secrets if available (used only by Section 8).
try:
    from google.colab import userdata
    try:
        v = userdata.get("HF_TOKEN")
        if v:
            os.environ["HF_TOKEN"] = v
    except Exception:
        pass
except ImportError:
    pass

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("HF login OK — Section 8 (Hub push) will work.")
else:
    print("HF_TOKEN not set — Section 8 (Hub push) will be skipped.")

print("\nConfig:")
for k in ("SENTINEL_URL", "MODEL_NAME", "SENTINEL_USE_VLLM",
          "SENTINEL_GRPO_NUM_GENERATIONS", "SENTINEL_GRPO_MAX_COMPLETION_LENGTH",
          "SENTINEL_GRPO_GRADIENT_ACCUMULATION_STEPS", "SENTINEL_GRPO_MAX_STEPS"):
    print(f"  {k:50s} = {os.environ.get(k)}")


## 2. Wake the SENTINEL OpenEnv on Hugging Face Spaces

SENTINEL ships as an [OpenEnv](https://github.com/meta-pytorch/OpenEnv) on
Hugging Face Spaces — a FastAPI service driven over HTTP. We use it for two
purposes:

* **Verification** — the cell below polls `/health` until the Space is
  warm (cold start ~60 s) and then hits `/reset` once to confirm the
  request/response shape matches what the trainer expects.
* **Reproducibility** — anyone reading this notebook can open the same Space
  in a browser to play scenarios manually in the Gradio replay viewer.

> **Implementation note for judges:** the GRPO training loop itself does
> *not* HTTP-call the Space on every step. `make_grpo_dataset` (Section 5)
> walks `server.environment.SentinelEnvironment` **in-process** to
> precompute one (prompt, ground_truth) row per Overseer decision. The
> reward function then grades each rollout in pure Python via
> `graders.grade_overseer_decision`. This keeps GRPO rollout latency
> reasonable on a free T4 — an HTTP round-trip per generation would
> otherwise dominate wall clock.


In [ ]:
from training.grpo_hf_job import warmup_sentinel, build_tool_env_cls, SENTINEL_URL

warmup_sentinel(SENTINEL_URL)

ToolEnv = build_tool_env_cls(SENTINEL_URL)
_env = ToolEnv()
first_obs = _env.reset(task_id="action_screen", seed=1)
print("First observation from /reset (truncated):\n")
print(first_obs[:600])


## 3. Load Qwen3-1.7B in 4-bit + attach a LoRA adapter

We fine-tune **only a LoRA adapter** (rank 16 on the four attention
projections). The base 1.7B Qwen3 stays frozen and 4-bit quantized — that
keeps the entire training loop under 16 GB on a T4.

`fast_inference=False` means we do **not** colocate vLLM with the trainer.
On T4 there's not enough VRAM for both the trainer's optimizer state and a
vLLM rollout engine — we use plain `model.generate(...)` instead, and the
GRPO smoke test in Section 5 confirms this still produces a learning signal.


In [ ]:
# Section 1.4 already did `import unsloth`; this just pulls the loader API.
from unsloth import FastLanguageModel
import torch

use_vllm = os.environ.get("SENTINEL_USE_VLLM", "0") == "1"

model, tokenizer = FastLanguageModel.from_pretrained(
    os.environ["MODEL_NAME"],
    max_seq_length=2048,    # T4-friendly. HF Jobs default is 4096.
    load_in_4bit=True,
    fast_inference=use_vllm,
)
print(f"Base model loaded. vLLM colocated = {use_vllm}")
print(f"VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")


### 3.1 *(Optional)* Zero-shot baseline F1

This runs the un-trained model against the full 50-scenario held-out split.
On T4 with HuggingFace generate (no vLLM) it takes ~10 minutes. The
baseline is **not required** for training — it's purely so you have a
"before" number to compare against the trained F1 in Section 7.

We skip it by default. Set `SENTINEL_RUN_ZEROSHOT_EVAL=1` and re-run this
cell if you want the comparison.


In [ ]:
from training.grpo_hf_job import _import_project, run_local_eval

project = _import_project()

baseline_f1 = {}
if os.environ.get("SENTINEL_RUN_ZEROSHOT_EVAL", "0") == "1":
    baseline_summary = run_local_eval(model, tokenizer, "qwen3_1_7b_zeroshot", project)
    baseline_f1 = baseline_summary["per_task_f1"]
    print("\nZero-shot per-tier F1:",
          {k: round(v["f1"], 3) for k, v in baseline_f1.items()})
else:
    print("Skipped (SENTINEL_RUN_ZEROSHOT_EVAL != '1').")
    print("The trained-vs-baseline plot in Section 7 will still render the "
          "naive / random / policy-aware reference baselines from eval_data/.")


### 3.2 Attach the LoRA adapter

Rank 16 on `q_proj`, `k_proj`, `v_proj`, `o_proj`. Trainable parameters:
~2 M (0.1% of the base model). Unsloth's gradient checkpointing cuts
activation memory by another ~30%.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"LoRA attached. Trainable: {trainable:,} / {total:,} "
      f"({100 * trainable / total:.3f}%)")


## 4. SFT warmup — teach the model the JSON output format

GRPO can only reinforce behaviour the model is already capable of producing.
Out of the box, Qwen3-1.7B doesn't emit the strict JSON shape
`{"decision": "...", "justification": "..."}` that the SENTINEL grader
expects, so we run **one epoch** of supervised fine-tuning on
`training/sft_data/sft_warmup.jsonl` first.

That file holds 2,623 (prompt, completion) pairs mined from the
policy-aware heuristic — the same baseline we're trying to beat. SFT here
is purely about format compliance; the GRPO step that follows is what
actually teaches the model **which** decision to pick.

On T4 this takes ~5 minutes for 1 epoch.


In [ ]:
from training.grpo_hf_job import run_sft

run_sft(model, tokenizer, epochs=1, output_dir="outputs/sft_warmup_1ep")
print("SFT warmup complete.")


## 5. GRPO smoke test (5 steps) — gate the long run

Before we burn an hour of T4 time on the long run, we run **5 GRPO steps**
to confirm:

1. Reward variance is non-zero (the binary grader fires at least once).
2. The policy isn't saturated (at least one log entry below 1.0 — otherwise
   GRPO has no signal to optimize against).
3. Per-step wall clock is under 90 s (otherwise the long run won't fit).

If any of those three fail, the assert at the end of this cell will fire
and the next sections won't run. The most common failure on T4 is "no
reward signal" — the fix is usually to re-run Section 4 with `epochs=2`.


In [ ]:
from pathlib import Path
from training.grpo_hf_job import (
    TrackingCallback, _build_grpo_trainer, make_grpo_dataset,
    GRPO_CONFIG, PLOTS_DIR, CKPT_DIR, SMOKE_STEPS,
)

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print("Building smoke dataset (n=64 proposals from action_screen seeds)...")
smoke_ds = make_grpo_dataset(n_samples=64)

smoke_cb = TrackingCallback(
    plots_dir=PLOTS_DIR,
    ckpt_dir=CKPT_DIR,
    model=model,
    plot_loss_fn=project["plot_loss"],
    plot_reward_fn=project["plot_reward"],
    plot_every=GRPO_CONFIG["save_steps"],
    is_smoke=True,
)
smoke_trainer = _build_grpo_trainer(
    model, tokenizer, smoke_ds, smoke_cb,
    output_dir="outputs/grpo_smoke",
    max_steps=SMOKE_STEPS,
    use_vllm=use_vllm,
)
smoke_trainer.train()

ok, msg = smoke_cb.smoke_pass()
print("\nSmoke result:", msg)
assert ok, "Smoke test failed. Re-run Section 4 with epochs=2 and try again."
print("Smoke OK — proceeding to the long run.")


## 6. GRPO training (60 steps default)

This is the actual training run. With T4 defaults (`max_steps=60`,
`num_generations=2`, `max_completion_length=256`,
`gradient_accumulation_steps=4`) it takes roughly **45 minutes** end to end.

Plots and checkpoints land in `training/plots/` and `training/checkpoints/`
every `SENTINEL_GRPO_SAVE_STEPS` steps. The notebook displays them inline
at the end of Section 7.

### Auto-abort safety net

`TrackingCallback` watches the rolling-mean reward and aborts early if
training stalls. The `STEP100_MIN_REWARD` / `STEP200_MIN_REWARD` thresholds
are read from env vars at module import time. On the T4 default of 60
steps, neither abort fires (we don't reach step 100), but the logic is
preserved so you can crank `SENTINEL_GRPO_MAX_STEPS=400` on L4/A100 and
get the auto-abort behaviour for free.


In [ ]:
n_long = GRPO_CONFIG["max_steps"]
print(f"Building long dataset (n={n_long * GRPO_CONFIG['gradient_accumulation_steps']} proposals)...")
long_ds = make_grpo_dataset(
    n_samples=n_long * GRPO_CONFIG["gradient_accumulation_steps"],
)

long_cb = TrackingCallback(
    plots_dir=PLOTS_DIR,
    ckpt_dir=CKPT_DIR,
    model=model,
    plot_loss_fn=project["plot_loss"],
    plot_reward_fn=project["plot_reward"],
    plot_every=GRPO_CONFIG["save_steps"],
)
long_trainer = _build_grpo_trainer(
    model, tokenizer, long_ds, long_cb,
    output_dir="outputs/grpo_long",
    max_steps=n_long,
    use_vllm=use_vllm,
)
long_trainer.train()

print(f"\nFinished {n_long} GRPO steps.")
print(f"Best reward window: {long_cb.best_reward:.3f} at step {long_cb.best_step}")
print(f"Abort path        : {long_cb.abort_reason or '(none — full run completed)'}")


## 7. Evaluate the trained adapter + render the comparison plot

We:

1. Save the trained LoRA adapter under
   `training/checkpoints/qwen3-1.7b-sentinel-best/`.
2. Run the full held-out eval (50 scenarios across `action_screen`,
   `war_room`, `drift_ops`).
3. Stack the result onto the per-baseline F1 numbers already in
   `eval_data/baseline_*.json` (naive, random, policy-aware, the published
   Qwen3 model, etc.) and render `baseline_vs_trained.png`.
4. Display all three plots inline so the judges don't need to dig.


In [ ]:
from training.grpo_hf_job import EVAL_DIR, _load_baselines

final_dir = CKPT_DIR / "qwen3-1.7b-sentinel-best"
final_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
print(f"Saved adapter -> {final_dir}")

print("\nRunning trained-model eval on 50 held-out scenarios...")
trained_summary = run_local_eval(
    model, tokenizer, "trained_qwen3_1_7b_grpo", project,
)
f1_per_tier = trained_summary["per_task_f1"]

baselines = _load_baselines(EVAL_DIR)
if baseline_f1:
    baselines["qwen3_1_7b_zeroshot"] = baseline_f1
baselines["trained_qwen3_1_7b_grpo"] = f1_per_tier

project["plot_baseline_vs_trained"](
    baselines,
    trained_label="trained_qwen3_1_7b_grpo",
    out_path=str(PLOTS_DIR / "baseline_vs_trained.png"),
    tier="action_screen",
)


In [ ]:
from IPython.display import Image, display, Markdown

display(Markdown("### Reward + loss curves"))
for name in ("grpo_loss.png", "grpo_reward.png"):
    p = PLOTS_DIR / name
    if p.exists():
        display(Image(filename=str(p)))

display(Markdown("### Trained vs. baseline F1 (action_screen tier)"))
p = PLOTS_DIR / "baseline_vs_trained.png"
if p.exists():
    display(Image(filename=str(p)))

display(Markdown("### Headline numbers"))
print(f"Zero-shot action_screen F1 : "
      f"{baseline_f1.get('action_screen', {}).get('f1', float('nan')):.3f}"
      f"  (skipped if Section 3.1 was off)")
print(f"Trained   action_screen F1 : {f1_per_tier['action_screen']['f1']:.3f}")
print(f"Trained   war_room     F1  : {f1_per_tier['war_room']['f1']:.3f}")
print(f"Trained   drift_ops    F1  : {f1_per_tier['drift_ops']['f1']:.3f}")


## 8. *(Optional)* Push the LoRA adapter to your HF Hub repo

Skip this section unless you've set `MODEL_REPO` to a repo you own and
added an `HF_TOKEN` with `write` scope to Colab Secrets. The default
`MODEL_REPO` (`Elliot89/sentinel-overseer-qwen3-1.7b`) is the published
hackathon checkpoint — you won't have permission to overwrite it.

To push to your own:

```python
os.environ["MODEL_REPO"] = "<your-username>/sentinel-overseer-qwen3-1.7b"
```

…then run the cell.


In [ ]:
from training.grpo_hf_job import _write_summary, push_lora_to_hub
import time

_write_summary(
    f1_per_tier=f1_per_tier,
    baseline_f1=baseline_f1,
    abort_path=long_cb.abort_reason,
    wall_clock_s=time.time(),
    best_step=long_cb.best_step,
)

if os.environ.get("HF_TOKEN"):
    url = push_lora_to_hub(final_dir)
    print(f"Adapter pushed -> {url}")
else:
    print("HF_TOKEN not set — Hub push skipped. Adapter still on disk at:")
    print(f"  {final_dir}")


---

### That's it

If you only care about the headline number, scroll up to the
**Trained vs. baseline F1** plot in Section 7. The full per-scenario JSON
for the trained run is in `eval_data/baseline_trained_qwen3_1_7b_grpo.json`,
and `training/run_summary.json` has the configuration that produced it.

**Repository:** [github.com/MrEinsteinE/sentinel-openenv](https://github.com/MrEinsteinE/sentinel-openenv)
**Live env demo:** [huggingface.co/spaces/Elliot89/sentinel](https://huggingface.co/spaces/Elliot89/sentinel)
**Published model:** [huggingface.co/Elliot89/sentinel-overseer-qwen3-1.7b](https://huggingface.co/Elliot89/sentinel-overseer-qwen3-1.7b)

Questions or repro issues? Open one on the GitHub repo above.
